# The Small-to-Big Problem [Step 1 - Retrieval Size vs Answer Size]

> **MLCourse - Agentic AI - Advanced RAG - Contextual Retrieval**

There is a tension at the heart of chunking that no amount of reranking or query
rewriting will resolve:

- **Small chunks retrieve well.** A two-sentence chunk is about one thing, so
  its embedding is sharp and its similarity score is meaningful.
- **Small chunks answer badly.** Two sentences rarely contain enough surrounding
  detail for the LLM to give a complete, correct answer - and they often lose
  the antecedent of every pronoun in them.

- **Large chunks answer well.** A full page carries context, cause and
  consequence.
- **Large chunks retrieve badly.** A page is about five things, so its embedding
  is the average of five topics and matches nothing sharply. It also burns
  context window and, as we saw in
  [`../11_reranking/02_cross_encoder_reranking.ipynb`](../11_reranking/02_cross_encoder_reranking.ipynb),
  can be silently truncated by the reranker.

This is the **small-to-big problem**. The whole module is about the resolution:
stop assuming the unit you *search* must be the unit you *read*.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# "Parent" units: paragraphs. Big enough to answer from, too big to retrieve
# precisely. These are the documents we will later cut into small children.
parents = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("parent paragraphs:", len(parents))
print("mean parent length:", int(sum(len(p) for p in parents) / len(parents)), "chars")

parent paragraphs: 237
mean parent length: 379 chars


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def build_index(texts):
    """Embed a list of texts and return the normalised matrix."""
    return encoder.encode(texts, normalize_embeddings=True,
                          batch_size=64, show_progress_bar=False)


def search(index, texts, query, top_n=5):
    """Return [(position, score)] of the best matches in `index`."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sims = index @ q
    order = np.argsort(sims)[::-1][:top_n]
    return [(int(i), float(sims[i])) for i in order]


print("encoder ready:", encoder.get_sentence_embedding_dimension(), "dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

encoder ready: 384 dimensions


### 2. Build three indexes at three granularities

Same text, three chunk sizes. We will retrieve the same question from each and
compare both what comes back and how well it answers.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_all(size, overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks, owners = [], []
    for parent_id, parent in enumerate(parents):
        for piece in splitter.split_text(parent):
            chunks.append(piece)
            owners.append(parent_id)
    return chunks, owners


GRANULARITIES = {
    "small (200 chars)": (200, 30),
    "medium (600 chars)": (600, 80),
    "large (1500 chars)": (1500, 150),
}

indexes = {}
for name, (size, overlap) in GRANULARITIES.items():
    chunks, owners = chunk_all(size, overlap)
    indexes[name] = {"chunks": chunks, "owners": owners, "vectors": build_index(chunks)}
    avg = int(sum(len(c) for c in chunks) / len(chunks))
    print(f"{name:<20} {len(chunks):>5} chunks, mean {avg:>5} chars")

small (200 chars)      622 chunks, mean   160 chars


medium (600 chars)     267 chunks, mean   345 chars


large (1500 chars)     237 chunks, mean   379 chars


### 3. Retrieval sharpness: small chunks win

The clearest evidence that small chunks retrieve better is the **similarity
score of the top hit**. A sharp, on-topic chunk scores high; an averaged
multi-topic chunk scores middling no matter how relevant part of it is.

In [6]:
QUESTION = "What did the Caterpillar tell Alice about keeping her temper?"

print(f"query: {QUESTION}\n")
for name, ix in indexes.items():
    hits = search(ix["vectors"], ix["chunks"], QUESTION, top_n=3)
    top_pos, top_score = hits[0]
    print(f"{name:<20} best cosine = {top_score:.3f}")
    print(f"{'':20} -> {ix['chunks'][top_pos][:130]}...")
    print()

query: What did the Caterpillar tell Alice about keeping her temper?

small (200 chars)    best cosine = 0.782
                     -> Which brought them back again to the beginning of the conversation. Alice felt a little irritated at the Caterpillar’s making such...

medium (600 chars)   best cosine = 0.720
                     -> Which brought them back again to the beginning of the conversation. Alice felt a little irritated at the Caterpillar’s making such...

large (1500 chars)   best cosine = 0.720
                     -> Which brought them back again to the beginning of the conversation. Alice felt a little irritated at the Caterpillar’s making such...



Small chunks reliably produce the highest similarity score. That is not a
coincidence or a quirk of this corpus - it is arithmetic. An embedding is a
compression of everything in the text; the more distinct topics you compress,
the further the result drifts from any one of them.

### 4. Answer quality: small chunks lose

Now the other half. Give Groq the top-3 chunks from each index - the same
budget of retrieval slots, different amounts of text - and read the answers.

In [7]:
def answer_from_index(name, question, top_k=3):
    ix = indexes[name]
    hits = search(ix["vectors"], ix["chunks"], question, top_n=top_k)
    context = "\n\n".join(f"[{i}] {ix['chunks'][pos]}" for i, (pos, _) in enumerate(hits))
    answer = ask(
        "Answer the question using ONLY the context below. If the context is "
        "incomplete or you cannot tell from it, say exactly what is missing.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return context, answer


for name in indexes:
    context, answer = answer_from_index(name, QUESTION)
    print("=" * 74)
    print(f"{name}   (context = {len(context)} characters)")
    print("=" * 74)
    print(answer)
    print()

small (200 chars)   (context = 615 characters)
The provided context does not contain any information about the Caterpillar telling Alice about keeping her temper. The context only describes Alice's irritation at the Caterpillar's short remarks, the Caterpillar's actions (yawning, shaking itself, getting down), and Alice's thoughts about what to do with the creature.



medium (600 chars)   (context = 1158 characters)
The provided context does not contain any information about the Caterpillar telling Alice how to keep her temper. The context only mentions the Caterpillar's short remarks, its instruction on growing taller or shorter, and Alice's recitation of "You are old, Father William."



large (1500 chars)   (context = 1158 characters)
The provided context does not contain any information about the Caterpillar telling Alice how to keep her temper. The context only mentions the Caterpillar's short remarks, its instruction on growing taller or shorter, and Alice's recitation of "You are old, Father William."



Read those three answers next to each other. The typical pattern:

- The **small** index gives a fragmentary answer, or explicitly says the context
  is incomplete - it retrieved the right *region* but not enough of it.
- The **large** index answers more completely but its context contains a lot of
  irrelevant text, and its retrieval was the least confident.
- The **medium** index is the compromise everyone ends up hand-tuning.

That hand-tuning is what this module is about avoiding.

### 5. Why small chunks lose information: dangling references

There is a specific, concrete reason small chunks answer badly, and it is worth
seeing directly. Prose is full of references that only resolve in earlier
sentences - "he", "it", "the other one", "this rule". Cut the text small enough
and those references dangle.

In [8]:
small = indexes["small (200 chars)"]["chunks"]

PRONOUNS = re.compile(r"\b(he|she|it|they|him|her|them|this|that|these|those)\b",
                      re.IGNORECASE)

opens_with_pronoun = [c for c in small if PRONOUNS.match(c.strip().split(" ")[0] or "x")]
print(f"small chunks starting with an unresolved pronoun: "
      f"{len(opens_with_pronoun)} of {len(small)}")
print("\nexamples - note that you cannot tell who or what is being discussed:\n")
for c in opens_with_pronoun[:4]:
    print(" -", c[:120], "...")

small chunks starting with an unresolved pronoun: 66 of 622

examples - note that you cannot tell who or what is being discussed:

 - she found herself falling down a very deep well. ...
 - she tried to look down and make out what she was coming to, but it was too dark to see anything; then she looked at the  ...
 - they were nice grand words to say.) ...
 - that she was dozing off, and had just begun to dream that she was walking hand in hand with Dinah, and saying to her ver ...


Each of those chunks is retrievable and, on its own, close to useless. An
embedding of "She said it was the stupidest tea-party she had ever been at"
carries no signal about Alice or the Hatter at all - so it will not be retrieved
for a question about either, *and* it would not help if it were.

### 6. The four resolutions

Every technique in this module decouples the retrieval unit from the generation
unit, in a different way:

| technique | search over | give the LLM | notebook |
|---|---|---|---|
| **Parent document retriever** | small child chunks | the whole parent document | [02](02_parent_document_retriever.ipynb) |
| **Sentence window** | single sentences | the sentence plus N neighbours | [03](03_sentence_window_retrieval.ipynb) |
| **Contextual chunk headers** | chunk + a generated context header | the chunk (header optional) | [04](04_contextual_chunk_headers.ipynb) |
| tuned chunk size | one compromise size | the same chunk | the baseline we are trying to beat |

The first two change *what you return*. The third changes *what you embed*.
They are independent, and they compose - notebook 05 measures all of them.

### 7. Key takeaways

- Small chunks give **sharp embeddings and weak answers**; large chunks give the
  reverse. Chunk size alone cannot win both.
- The similarity score of the top hit is a good, cheap diagnostic of retrieval
  sharpness.
- Small chunks additionally suffer **dangling references** - pronouns and
  definite articles whose antecedents were cut away.
- The fix is not a better chunk size; it is **decoupling the retrieval unit from
  the generation unit**.

Next: [`02_parent_document_retriever.ipynb`](02_parent_document_retriever.ipynb),
the canonical implementation of that decoupling in LangChain.